In [1]:
from pathlib import Path
import time

import numpy as np
import tensorflow as tf

from google.colab import drive


# MOUNT GOOGLE DRIVE
drive.mount("/content/drive")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Temporal MLP for Environmental Noise Cancellation

# 1. IMPORTS
import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Reshape,
    Add,
    LayerNormalization,
    Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

# 2. EXPERIMENT CONFIGURATION
EXPERIMENT_NAME = "TMLP_baseline"

SEED = 42

SAMPLE_RATE = 16000

WINDOW_SIZE = 256
HOP_SIZE = 128

BATCH_SIZE = 32
EPOCHS = 10

# IMPORTANT:
# Reduced from the previous 0.01 to 0.001
LEARNING_RATE = 0.001

# Gradient clipping for training stability
CLIPNORM = 1.0

# 3. T-MLP ARCHITECTURE CONFIGURATION
PROJECTION_DIM = 64
HIDDEN_DIM = 64

DROPOUT_RATE = 0.1

NUM_TMLP_BLOCKS = 2

# 4. PATH CONFIGURATION
DATASET_PATH = "/content/drive/MyDrive/T-MLP/processed_data"

TRAIN_PATH = os.path.join(
    DATASET_PATH,
    "train"
)

VAL_PATH = os.path.join(
    DATASET_PATH,
    "validation"
)

TEST_PATH = os.path.join(
    DATASET_PATH,
    "test"
)


# Model directory
MODEL_DIR = "/content/drive/MyDrive/T-MLP/saved_models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "TMLP_best.keras"
)

# 5. REPRODUCIBILITY
np.random.seed(SEED)

tf.random.set_seed(SEED)

# 6. ENVIRONMENT INFORMATION
print("\n" + "=" * 60)
print("T-MLP ENVIRONMENT")
print("=" * 60)

print(f"Experiment: {EXPERIMENT_NAME}")

print(
    f"TensorFlow: {tf.__version__}"
)

print(
    f"Random seed: {SEED}"
)


# GPU information
print("\nAvailable GPUs:")

gpus = tf.config.list_physical_devices(
    "GPU"
)

if gpus:

    for gpu in gpus:
        print(f"  {gpu}")

else:

    print("  No GPU detected.")


# Deterministic operations
try:

    tf.config.experimental.enable_op_determinism()

    print(
        "\nDeterministic operations: ENABLED"
    )

except Exception:

    print(
        "\nDeterministic operations: "
        "Could not be enabled."
    )

# 7. DATASET VERIFICATION
def get_npz_files(folder):

    """
    Return sorted list of .npz files.
    """

    if not os.path.exists(folder):

        raise FileNotFoundError(
            f"Folder not found:\n{folder}"
        )

    files = [
        os.path.join(folder, file)
        for file in os.listdir(folder)
        if file.endswith(".npz")
    ]

    return sorted(files)


train_files = get_npz_files(
    TRAIN_PATH
)

val_files = get_npz_files(
    VAL_PATH
)

test_files = get_npz_files(
    TEST_PATH
)


print("\n" + "=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

print(
    f"Training samples   : "
    f"{len(train_files)}"
)

print(
    f"Validation samples : "
    f"{len(val_files)}"
)

print(
    f"Test samples       : "
    f"{len(test_files)}"
)


# Fixed dataset verification
if len(train_files) != 800:

    raise ValueError(
        f"Expected 800 training samples, "
        f"found {len(train_files)}"
    )


if len(val_files) != 100:

    raise ValueError(
        f"Expected 100 validation samples, "
        f"found {len(val_files)}"
    )


if len(test_files) != 100:

    raise ValueError(
        f"Expected 100 test samples, "
        f"found {len(test_files)}"
    )


print(
    "\nDataset verification successful."
)

# 8. LOAD NPZ SAMPLE
def load_npz_sample(file_path):

    """
    Load noisy and clean speech from
    one .npz file.
    """

    data = np.load(
        file_path
    )

    noisy = data[
        "noisy_speech"
    ].astype(
        np.float32
    )

    clean = data[
        "clean_speech"
    ].astype(
        np.float32
    )

    return noisy, clean

# 9. CREATE SLIDING WINDOWS
def create_windows(
    noisy,
    clean
):

    
    # Converting audio into overlapping windows using sliding window approach
    noisy_windows = []

    clean_windows = []


    if len(noisy) != len(clean):

        raise ValueError(
            "Noisy and clean audio "
            "lengths do not match."
        )


    if len(noisy) < WINDOW_SIZE:

        return (
            np.empty(
                (
                    0,
                    WINDOW_SIZE
                ),
                dtype=np.float32
            ),

            np.empty(
                (
                    0,
                    WINDOW_SIZE
                ),
                dtype=np.float32
            )
        )


    for start in range(
        0,
        len(noisy) - WINDOW_SIZE + 1,
        HOP_SIZE
    ):

        end = start + WINDOW_SIZE

        noisy_window = noisy[
            start:end
        ]

        clean_window = clean[
            start:end
        ]

        noisy_windows.append(
            noisy_window
        )

        clean_windows.append(
            clean_window
        )


    return (

        np.asarray(
            noisy_windows,
            dtype=np.float32
        ),

        np.asarray(
            clean_windows,
            dtype=np.float32
        )
    )

# 10. LOAD DATASET
def load_dataset(
    file_list,
    dataset_name
):
    print(
        f"\nLoading {dataset_name} dataset..."
    )


    all_noisy = []

    all_clean = []


    for index, file_path in enumerate(
        file_list
    ):

        noisy, clean = load_npz_sample(
            file_path
        )


        noisy_windows, clean_windows = (
            create_windows(
                noisy,
                clean
            )
        )


        if len(noisy_windows) == 0:

            print(
                f"Warning: skipped "
                f"{os.path.basename(file_path)} "
                f"(too short)"
            )

            continue


        all_noisy.append(
            noisy_windows
        )

        all_clean.append(
            clean_windows
        )


        if (index + 1) % 100 == 0:

            print(
                f"Loaded "
                f"{index + 1}/"
                f"{len(file_list)} files"
            )


    if not all_noisy:

        raise ValueError(
            f"No valid windows found "
            f"in {dataset_name} dataset."
        )


    X = np.concatenate(
        all_noisy,
        axis=0
    )

    y = np.concatenate(
        all_clean,
        axis=0
    )


    print(
        f"{dataset_name} windows: "
        f"{len(X)}"
    )


    return X, y

# 11. LOAD TRAIN / VALIDATION / TEST
print("\n" + "=" * 60)
print("LOADING TRAINING DATA")
print("=" * 60)

X_train, y_train = load_dataset(
    train_files,
    "training"
)


print("\n" + "=" * 60)
print("LOADING VALIDATION DATA")
print("=" * 60)

X_val, y_val = load_dataset(
    val_files,
    "validation"
)


print("\n" + "=" * 60)
print("LOADING TEST DATA")
print("=" * 60)

X_test, y_test = load_dataset(
    test_files,
    "test"
)

# 12. NUMERICAL VALIDATION
print("\n" + "=" * 60)
print("DATA NUMERICAL VALIDATION")
print("=" * 60)


def check_array(
    name,
    array
):

    print(
        f"{name}: "
        f"shape={array.shape}, "
        f"dtype={array.dtype}"
    )

    if np.isnan(array).any():

        raise ValueError(
            f"{name} contains NaN values."
        )

    if np.isinf(array).any():

        raise ValueError(
            f"{name} contains Inf values."
        )


check_array(
    "X_train",
    X_train
)

check_array(
    "y_train",
    y_train
)

check_array(
    "X_val",
    X_val
)

check_array(
    "y_val",
    y_val
)

check_array(
    "X_test",
    X_test
)

check_array(
    "y_test",
    y_test
)


print(
    "\nNaN/Inf validation: PASSED"
)

# 13. DATASET SUMMARY
print("\n" + "=" * 60)
print("WINDOWED DATASET")
print("=" * 60)

print(
    f"Training windows   : "
    f"{X_train.shape}"
)

print(
    f"Validation windows : "
    f"{X_val.shape}"
)

print(
    f"Test windows       : "
    f"{X_test.shape}"
)

print(
    f"\nInput window size  : "
    f"{WINDOW_SIZE} samples"
)

print(
    f"Sample rate        : "
    f"{SAMPLE_RATE} Hz"
)

print(
    f"Window duration    : "
    f"{WINDOW_SIZE / SAMPLE_RATE * 1000:.2f} ms"
)

print(
    f"Hop size           : "
    f"{HOP_SIZE} samples"
)

# 14. TEMPORAL MLP BLOCK
class TemporalMLPBlock(
    tf.keras.layers.Layer
):

    """
    Temporal MLP block.

    Processing:

        Dense
          ↓
        ReLU
          ↓
        Dropout
          ↓
        Dense
          ↓
        Residual addition
          ↓
        LayerNorm
    """


    def __init__(
        self,
        hidden_dim=64,
        dropout_rate=0.1,
        **kwargs
    ):

        super().__init__(
            **kwargs
        )


        self.dense_1 = Dense(
            hidden_dim,
            activation="relu"
        )


        self.dropout = Dropout(
            dropout_rate
        )


        self.dense_2 = Dense(
            hidden_dim,
            activation=None
        )


        self.add = Add()


        self.layer_norm = (
            LayerNormalization()
        )


    def call(
        self,
        inputs,
        training=None
    ):

        x = self.dense_1(
            inputs
        )


        x = self.dropout(
            x,
            training=training
        )


        x = self.dense_2(
            x
        )


        x = self.add(
            [
                inputs,
                x
            ]
        )


        x = self.layer_norm(
            x
        )


        return x

# 15. BUILD T-MLP MODEL
def build_tmlp():

    inputs = Input(
        shape=(WINDOW_SIZE,),
        name="noisy_audio"
    )

    # Convert waveform vector into temporal representation
    x = Reshape(
        (
            WINDOW_SIZE,
            1
        ),
        name="temporal_reshape"
    )(inputs)

    # Feature projection
    x = Dense(
        PROJECTION_DIM,
        activation="relu",
        name="feature_projection"
    )(x)

    # Temporal MLP Block 1
    x = TemporalMLPBlock(
        hidden_dim=HIDDEN_DIM,
        dropout_rate=DROPOUT_RATE,
        name="temporal_mlp_block_1"
    )(x)

    # Temporal MLP Block 2
    x = TemporalMLPBlock(
        hidden_dim=HIDDEN_DIM,
        dropout_rate=DROPOUT_RATE,
        name="temporal_mlp_block_2"
    )(x)

    # Feature reduction
    x = Dense(
        32,
        activation="relu",
        name="feature_reduction"
    )(x)

    # Generate one output per time step
    x = Dense(
        1,
        activation="linear",
        name="output_projection"
    )(x)

    # Convert back to waveform
    outputs = Reshape(
        (
            WINDOW_SIZE,
        ),
        name="output_waveform"
    )(x)


    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="T_MLP"
    )

    # Optimizer
    optimizer = Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=CLIPNORM
    )

    # Compile
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )


    return model

# 16. CREATE MODEL
print("\n" + "=" * 60)
print("BUILDING T-MLP")
print("=" * 60)


model = build_tmlp()


model.summary()


print(
    "\nTotal parameters: "
    f"{model.count_params():,}"
)

# 17. CONFIGURATION SUMMARY
print("\n" + "=" * 60)
print("T-MLP TRAINING CONFIGURATION")
print("=" * 60)

print(
    f"Epochs             : {EPOCHS}"
)

print(
    f"Batch size         : {BATCH_SIZE}"
)

print(
    f"Learning rate      : {LEARNING_RATE}"
)

print(
    f"Gradient clipnorm  : {CLIPNORM}"
)

print(
    f"Window size        : {WINDOW_SIZE}"
)

print(
    f"Hop size           : {HOP_SIZE}"
)

print(
    f"Projection dim     : {PROJECTION_DIM}"
)

print(
    f"Hidden dim         : {HIDDEN_DIM}"
)

print(
    f"T-MLP blocks       : {NUM_TMLP_BLOCKS}"
)

print(
    f"Dropout rate       : {DROPOUT_RATE}"
)

print(
    f"Best model path    : {BEST_MODEL_PATH}"
)

# 18. CONFIGURE BEST-MODEL CHECKPOINT
print("\n" + "=" * 60)
print("CONFIGURING CHECKPOINT")
print("=" * 60)


checkpoint = ModelCheckpoint(

    filepath=BEST_MODEL_PATH,

    monitor="val_loss",

    mode="min",

    save_best_only=True,

    save_weights_only=False,

    verbose=1
)


print(
    "Monitoring: val_loss"
)

print(
    "Mode: minimize"
)

print(
    "Save best only: YES"
)

# 19. TRAIN T-MLP
print("\n" + "=" * 60)
print("STARTING T-MLP TRAINING")
print("=" * 60)


training_start = time.perf_counter()


history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    shuffle=True,

    callbacks=[
        checkpoint
    ],

    verbose=1
)


training_end = time.perf_counter()


training_time = (
    training_end -
    training_start
)


print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)

print(
    f"Epochs trained: "
    f"{len(history.history['loss'])}"
)

# 20. LOAD BEST T-MLP MODEL
print("\n" + "=" * 60)
print("LOADING BEST T-MLP MODEL")
print("=" * 60)


if not os.path.exists(
    BEST_MODEL_PATH
):

    raise FileNotFoundError(
        "Best T-MLP model was not created."
    )


best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={
        "TemporalMLPBlock":
        TemporalMLPBlock
    }
)


print(
    "Best T-MLP model loaded successfully."
)

print(
    f"Path: {BEST_MODEL_PATH}"
)


print(
    f"Parameters: "
    f"{best_model.count_params():,}"
)

# 21. TEST EVALUATION
print("\n" + "=" * 60)
print("TEST EVALUATION")
print("=" * 60)


test_loss, test_mae = (
    best_model.evaluate(

        X_test,
        y_test,

        batch_size=BATCH_SIZE,

        verbose=1
    )
)


test_rmse = np.sqrt(
    test_loss
)


print(
    f"\nTest MSE  : "
    f"{test_loss:.8f}"
)

print(
    f"Test RMSE : "
    f"{test_rmse:.8f}"
)

print(
    f"Test MAE  : "
    f"{test_mae:.8f}"
)

# 22. GENERATE TEST PREDICTIONS
print("\n" + "=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)


prediction_start = time.perf_counter()


predictions = best_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)


prediction_end = time.perf_counter()


prediction_time = (
    prediction_end -
    prediction_start
)


print(
    f"Prediction time: "
    f"{prediction_time:.4f} seconds"
)

print(
    f"Prediction shape: "
    f"{predictions.shape}"
)

# 23. PREDICTION VALIDATION
print("\n" + "=" * 60)
print("PREDICTION VALIDATION")
print("=" * 60)


prediction_nan = np.isnan(
    predictions
).any()


prediction_inf = np.isinf(
    predictions
).any()


if prediction_nan or prediction_inf:

    raise ValueError(
        "Predictions contain NaN or Inf."
    )


print(
    "NaN/Inf check: PASSED"
)


print(
    f"Prediction min : "
    f"{predictions.min():.6f}"
)

print(
    f"Prediction max : "
    f"{predictions.max():.6f}"
)

print(
    f"Prediction mean: "
    f"{predictions.mean():.6f}"
)

print(
    f"Prediction std : "
    f"{predictions.std():.6f}"
)

# 24. INFERENCE LATENCY
print("\n" + "=" * 60)
print("MEASURING INFERENCE LATENCY")
print("=" * 60)


# Warm-up

_ = best_model.predict(
    X_test[:1],
    verbose=0
)


latencies = []


num_latency_samples = min(
    100,
    len(X_test)
)


for i in range(
    num_latency_samples
):

    sample = X_test[
        i:i + 1
    ]


    start = time.perf_counter()


    _ = best_model.predict(
        sample,
        verbose=0
    )


    end = time.perf_counter()


    latency_ms = (
        end - start
    ) * 1000


    latencies.append(
        latency_ms
    )


latencies = np.asarray(
    latencies
)


mean_latency = np.mean(
    latencies
)

median_latency = np.median(
    latencies
)

min_latency = np.min(
    latencies
)

max_latency = np.max(
    latencies
)


print(
    f"Samples measured : "
    f"{num_latency_samples}"
)

print(
    f"Mean latency     : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median latency   : "
    f"{median_latency:.4f} ms"
)

print(
    f"Minimum latency  : "
    f"{min_latency:.4f} ms"
)

print(
    f"Maximum latency  : "
    f"{max_latency:.4f} ms"
)

# 25. MODEL SIZE
print("\n" + "=" * 60)
print("MODEL INFORMATION")
print("=" * 60)


model_size_bytes = os.path.getsize(
    BEST_MODEL_PATH
)


model_size_kb = (
    model_size_bytes / 1024
)


model_size_mb = (
    model_size_bytes /
    (1024 ** 2)
)


print(
    f"Parameters : "
    f"{best_model.count_params():,}"
)

print(
    f"Model size : "
    f"{model_size_kb:.2f} KB"
)

print(
    f"Model size : "
    f"{model_size_mb:.4f} MB"
)

# 26. FINAL RESULTS
print("\n" + "=" * 60)
print("T-MLP FINAL RESULTS")
print("=" * 60)


print(
    f"Parameters       : "
    f"{best_model.count_params():,}"
)

print(
    f"MSE              : "
    f"{test_loss:.8f}"
)

print(
    f"RMSE             : "
    f"{test_rmse:.8f}"
)

print(
    f"MAE              : "
    f"{test_mae:.8f}"
)

print(
    f"Mean Latency     : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median Latency   : "
    f"{median_latency:.4f} ms"
)

print(
    f"Model Size       : "
    f"{model_size_kb:.2f} KB"
)

print(
    f"Training Time    : "
    f"{training_time:.2f} seconds"
)

print(
    f"Best Model       : "
    f"{BEST_MODEL_PATH}"
)

print("=" * 60)
print("T-MLP TRAINING COMPLETE")
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

T-MLP ENVIRONMENT
Experiment: TMLP_baseline
TensorFlow: 2.21.0
Random seed: 42

Available GPUs:
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Deterministic operations: ENABLED

DATASET VERIFICATION
Training samples   : 800
Validation samples : 100
Test samples       : 100

Dataset verification successful.

LOADING TRAINING DATA

Loading training dataset...
Loaded 100/800 files
Loaded 200/800 files
Loaded 300/800 files
Loaded 400/800 files
Loaded 500/800 files
Loaded 600/800 files
Loaded 700/800 files
Loaded 800/800 files
training windows: 732388

LOADING VALIDATION DATA

Loading validation dataset...
Loaded 100/100 files
validation windows: 80583

LOADING TEST DATA

Loading test dataset...
Loaded 100/100 files
test windows: 85649

DATA NUMERICAL VALIDATION
X_train: shape=(732388, 256), dtype=float32
y_train: shape=(732388, 256), dtype=f

Model: "T_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ noisy_audio (InputLayer)        │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_reshape (Reshape)      │ (None, 256, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_projection (Dense)      │ (None, 256, 64)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_mlp_block_1            │ (None, 256, 64)        │         8,448 │
│ (TemporalMLPBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_mlp_block_2            │ (None, 256, 64)        │         8,448 │
│ (TemporalMLPBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_reduction (Dense)       │ (None, 256, 32)        │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_projection (Dense)       │ (None, 256, 1)         │            33 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_waveform (Reshape)       │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,137 (74.75 KB)

 Trainable params: 19,137 (74.75 KB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 19,137

T-MLP TRAINING CONFIGURATION
Epochs             : 10
Batch size         : 32
Learning rate      : 0.001
Gradient clipnorm  : 1.0
Window size        : 256
Hop size           : 128
Projection dim     : 64
Hidden dim         : 64
T-MLP blocks       : 2
Dropout rate       : 0.1
Best model path    : /content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras

CONFIGURING CHECKPOINT
Monitoring: val_loss
Mode: minimize
Save best only: YES

STARTING T-MLP TRAINING
Epoch 1/10
22887/22888 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0027 - mae: 0.0276
Epoch 1: val_loss improved from None to 0.00128, saving model to /content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras
22888/22888 ━━━━━━━━━━━━━━━━━━━━ 247s 10ms/step - loss: 0.0019 - mae: 0.0255 - val_loss: 0.0013 - val_mae: 0.0207
Epoch 2/10
22883/22888 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0018 - mae: 0.0246
Epoch 2: val

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'temporal_mlp_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'temporal_mlp_block_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Best T-MLP model loaded successfully.
Path: /content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras
Parameters: 19,137

TEST EVALUATION
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.0013 - mae: 0.0207

Test MSE  : 0.00129801
Test RMSE : 0.03602786
Test MAE  : 0.02067654

GENERATING TEST PREDICTIONS
2677/2677 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step
Prediction time: 8.4492 seconds
Prediction shape: (85649, 256)

PREDICTION VALIDATION
NaN/Inf check: PASSED
Prediction min : -0.243973
Prediction max : 0.233260
Prediction mean: -0.000735
Prediction std : 0.040654

MEASURING INFERENCE LATENCY
Samples measured : 100
Mean latency     : 92.6122 ms
Median latency   : 81.1424 ms
Minimum latency  : 75.4612 ms
Maximum latency  : 147.3384 ms

MODEL INFORMATION
Parameters : 19,137
Model size : 292.70 KB
Model size : 0.2858 MB

T-MLP FINAL RESULTS
Parameters       : 19,137
MSE              : 0.00129801
RMSE             : 0.03602786
MAE              : 0.02067654
Mean Latency     : 92.6122 ms
Median L